# 🇬🇧 UK Retail Performance Hub
## Phase 1 | Notebook 2 of 4 — Data Cleaning & Transformation

**What this notebook does:**
- Loads the raw UCI and ONS data from Google Drive
- Removes invalid rows (cancellations, nulls, test codes)
- Adds new columns: Revenue, Month, Year, Product Category, Customer Cohort
- Saves clean data as parquet files ready for the database

**Prerequisites:** Run `01_ingest.ipynb` first

---

### Cell 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
ROOT          = Path('/content/drive/MyDrive/uk-retail-performance-hub')
RAW_DIR       = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print('✓ Drive mounted')
print(f'  Raw data:       {RAW_DIR}')
print(f'  Processed data: {PROCESSED_DIR}')

### Cell 2 — Import libraries

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.float_format', '{:,.2f}'.format)
print('✓ Libraries imported')

### Cell 3 — Load raw UCI data
This reads the full Excel file. It will take about 30–60 seconds as it's ~24MB.

In [ ]:
uci_path = RAW_DIR / 'online_retail.xlsx'

print('Loading UCI data (this takes ~60 seconds)...')
df = pd.read_excel(uci_path, dtype={'CustomerID': str})

print(f'\n✓ Loaded successfully')
print(f'  Rows     : {len(df):,}')
print(f'  Columns  : {list(df.columns)}')
print(f'\nSample data:')
df.head(3)

### Cell 4 — Data quality check (before cleaning)
Shows you exactly what needs fixing before we touch anything.

In [ ]:
print('RAW DATA QUALITY REPORT')
print('=' * 45)
print(f'Total rows             : {len(df):>10,}')
print(f'Null CustomerIDs       : {df["CustomerID"].isna().sum():>10,}')
print(f'Cancelled orders (C*)  : {df["InvoiceNo"].astype(str).str.startswith("C").sum():>10,}')
print(f'Negative quantities    : {(df["Quantity"] <= 0).sum():>10,}')
print(f'Zero/negative prices   : {(df["UnitPrice"] <= 0).sum():>10,}')
print(f'Unique countries       : {df["Country"].nunique():>10,}')
print(f'Date range             : {df["InvoiceDate"].min()} → {df["InvoiceDate"].max()}')

### Cell 5 — Clean the UCI data
Removes invalid rows step by step. Each step prints how many rows were removed so you can see exactly what changed.

In [ ]:
raw_count = len(df)
print(f'Starting rows: {raw_count:,}\n')

# Step 1: Remove cancellations
cancelled = df['InvoiceNo'].astype(str).str.startswith('C')
df = df[~cancelled]
print(f'After removing cancellations  : {len(df):,} rows  (removed {cancelled.sum():,})')

# Step 2: Remove rows with no CustomerID
before = len(df)
df = df[df['CustomerID'].notna()]
print(f'After removing null customers : {len(df):,} rows  (removed {before - len(df):,})')

# Step 3: Remove invalid quantities and prices
before = len(df)
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]
print(f'After removing invalid qty/px : {len(df):,} rows  (removed {before - len(df):,})')

# Step 4: Remove non-product stock codes (letters only = adjustments/postage)
before = len(df)
test_codes = df['StockCode'].astype(str).str.fullmatch(r'[A-Za-z]+')
df = df[~test_codes]
print(f'After removing test codes     : {len(df):,} rows  (removed {before - len(df):,})')

print(f'\n✓ Cleaning complete')
print(f'  Kept {len(df)/raw_count*100:.1f}% of original data ({len(df):,} of {raw_count:,} rows)')

### Cell 6 — Add derived columns
Creates all the new columns Power BI will use: Revenue, dates, UK flag, Cohort Month.

In [ ]:
print('Adding derived columns...')

# Ensure InvoiceDate is datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Revenue
df['Revenue'] = (df['Quantity'] * df['UnitPrice']).round(2)

# Date breakdowns
df['Year']      = df['InvoiceDate'].dt.year
df['Month']     = df['InvoiceDate'].dt.month
df['YearMonth'] = df['InvoiceDate'].dt.to_period('M').astype(str)
df['Quarter']   = df['InvoiceDate'].dt.to_period('Q').astype(str)
df['DayOfWeek'] = df['InvoiceDate'].dt.day_name()
df['IsWeekend'] = df['InvoiceDate'].dt.dayofweek >= 5

# UK vs International flag
df['IsUK'] = df['Country'].str.strip().str.upper() == 'UNITED KINGDOM'

# Customer cohort month (the month they first purchased)
first_purchase = (
    df.groupby('CustomerID')['InvoiceDate']
    .min()
    .dt.to_period('M')
    .astype(str)
    .rename('CohortMonth')
)
df = df.merge(first_purchase, on='CustomerID', how='left')

# Clean CustomerID whitespace
df['CustomerID'] = df['CustomerID'].str.strip()

print('\n✓ Columns added:')
print('  Revenue, Year, Month, YearMonth, Quarter, DayOfWeek, IsWeekend, IsUK, CohortMonth')
print(f'\nTotal revenue in dataset: £{df["Revenue"].sum():,.2f}')
print(f'Unique customers        : {df["CustomerID"].nunique():,}')
print(f'Unique products         : {df["StockCode"].nunique():,}')

### Cell 7 — Product category classification
Groups products into categories using keywords in the product description.
This takes about 30 seconds.

In [ ]:
CATEGORY_KEYWORDS = {
    'Christmas & Seasonal'  : ['christmas','xmas','santa','reindeer','advent','tinsel','bauble','wreath','snowflake','noel'],
    'Candles & Lighting'    : ['candle','lantern','tealight','tea light','nightlight','lamp','light','fairy light'],
    'Bags & Accessories'    : ['bag','tote','purse','wallet','satchel','handbag','pouch'],
    'Home Décor'            : ['frame','mirror','clock','wall','door','cushion','pillow','bunting','sign','plaque','garland','banner'],
    'Kitchenware & Dining'  : ['mug','cup','bowl','plate','tin','jar','jug','bottle','spoon','napkin','coaster','tray'],
    'Stationery & Cards'    : ['card','notebook','diary','pen','pencil','sticker','label','wrap','ribbon','tag'],
    'Toys & Games'          : ['toy','game','doll','ball','puppet','puzzle','play'],
    'Novelty & Gifts'       : ['gift','novel','fun','party','balloon','charm','badge'],
}

def categorise_product(description):
    if not isinstance(description, str):
        return 'Uncategorised'
    desc_lower = description.lower()
    for category, keywords in CATEGORY_KEYWORDS.items():
        if any(kw in desc_lower for kw in keywords):
            return category
    return 'Uncategorised'

print('Categorising products...')
df['ProductCategory'] = df['Description'].apply(categorise_product)

print('\n✓ Category distribution:')
cat_summary = df.groupby('ProductCategory')['Revenue'].agg(['count','sum']).rename(columns={'count':'Rows','sum':'Revenue'})
cat_summary['Revenue'] = cat_summary['Revenue'].map('£{:,.0f}'.format)
print(cat_summary.sort_values('Rows', ascending=False).to_string())

### Cell 8 — Clean ONS data
Loads and aligns the ONS Retail Sales Index to the same date range as the UCI data.

In [ ]:
ons_path = RAW_DIR / 'ons_rsi.csv'

if not ons_path.exists():
    print('⚠ ONS file not found — skipping. The market benchmark page in Power BI will be empty.')
    print('  Download manually from: https://www.ons.gov.uk/businessindustryandtrade/retailindustry/datasets/retailsales')
    ons_clean = None
else:
    print('Loading ONS data...')
    raw_ons = pd.read_csv(ons_path, header=None, encoding='latin-1')

    # Find the header row — ONS files have metadata rows at the top
    header_row = 0
    for i, row in raw_ons.iterrows():
        if row.astype(str).str.contains(r'\b(19|20)\d{2}\b', regex=True).any():
            header_row = i
            break

    ons = pd.read_csv(ons_path, skiprows=header_row, encoding='latin-1')

    # Take first two columns as Period and RSI_Value
    ons = ons.iloc[:, :2].copy()
    ons.columns = ['Period', 'RSI_Value']
    ons = ons.dropna()

    ons['Period']    = pd.to_datetime(ons['Period'], infer_datetime_format=True, errors='coerce')
    ons['RSI_Value'] = pd.to_numeric(ons['RSI_Value'], errors='coerce')
    ons = ons.dropna().sort_values('Period').reset_index(drop=True)

    # Filter to UCI date range
    uci_start = df['InvoiceDate'].min()
    uci_end   = df['InvoiceDate'].max()
    ons_clean = ons[(ons['Period'] >= uci_start) & (ons['Period'] <= uci_end)].copy()

    # Add derived columns
    ons_clean['YearMonth']   = ons_clean['Period'].dt.to_period('M').astype(str)
    ons_clean['Year']        = ons_clean['Period'].dt.year
    ons_clean['Month']       = ons_clean['Period'].dt.month
    ons_clean['RSI_MoM_Pct'] = ons_clean['RSI_Value'].pct_change().mul(100).round(2)

    print(f'✓ ONS data cleaned')
    print(f'  Rows in date range: {len(ons_clean)}')
    print(ons_clean[['YearMonth','RSI_Value','RSI_MoM_Pct']].to_string(index=False))

### Cell 9 — Save cleaned data to Google Drive
Saves both datasets as parquet files — a fast, compact format that loads quickly into the database.

In [ ]:
# Convert boolean columns to int (parquet compatibility)
df['IsUK']      = df['IsUK'].astype(int)
df['IsWeekend'] = df['IsWeekend'].astype(int)

# Save UCI clean data
uci_out = PROCESSED_DIR / 'uci_clean.parquet'
df.to_parquet(uci_out, index=False)
size_mb = uci_out.stat().st_size / 1_000_000
print(f'✓ UCI clean data saved  ({size_mb:.1f} MB)')
print(f'  Path: {uci_out}')

# Save ONS clean data
if ons_clean is not None and len(ons_clean) > 0:
    ons_out = PROCESSED_DIR / 'ons_clean.parquet'
    ons_clean.to_parquet(ons_out, index=False)
    print(f'\n✓ ONS clean data saved')
    print(f'  Path: {ons_out}')

print('\n' + '=' * 55)
print('  ✓ Cleaning complete — run Notebook 03_load_db.ipynb next')
print('=' * 55)

### Cell 10 — Final data preview
A quick look at the clean dataset before handing it to the database.

In [ ]:
print('CLEAN DATA SUMMARY')
print('=' * 45)
print(f'Total rows        : {len(df):,}')
print(f'Total revenue     : £{df["Revenue"].sum():,.2f}')
print(f'Unique customers  : {df["CustomerID"].nunique():,}')
print(f'Unique products   : {df["StockCode"].nunique():,}')
print(f'Date range        : {df["InvoiceDate"].min().date()} → {df["InvoiceDate"].max().date()}')
print(f'UK revenue share  : {df[df["IsUK"]==1]["Revenue"].sum() / df["Revenue"].sum() * 100:.1f}%')
print(f'\nMonthly revenue:')
df.groupby('YearMonth')['Revenue'].sum().map('£{:,.0f}'.format)